In [1]:
import os
import pandas as pd
import base64
from openai import OpenAI
import re
import sqlite3
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime
import sys
import time
from openai import APIStatusError, RateLimitError, InternalServerError
from PIL import Image

# Vergleich Gold-Standard (manuell) vs. LLM-generiert so zusammenfassen:

1. Clarity (Verständlichkeit)

Tendenz: LLM-Texte sind oft sehr klar formuliert (LLM-as-judge häufig 4–5), weil sie konsistent strukturiert sind (Metadata → Overview → Long description). Gold-Standard ist ebenfalls klar, wirkt aber häufig prägnanter und stärker auf das Wesentliche fokussiert.

Beispiel, wo LLM klar ist:
Beim MWSt-Chart beschreibt ein LLM-Text die Entwicklung mit konkreten Jahreswerten in sauberer Reihenfolge (1995→2030).

Beispiel, wo Gold-Standard oft „klarer durch Fokus“ wirkt:
Beim Kakao-Chart steht als Kernaussage: Côte d’Ivoire produziert mindestens doppelt so viel wie alle anderen einzeln – das ist extrem leicht zu „verstehen“, weil es die Hauptbotschaft direkt liefert.

Kritischer Punkt: LLM-Klarheit kann trügen, wenn scheinbar plausible Zusatz-Erklärungen auftauchen (z.B. politische/kausale Begründungen), die nicht eindeutig aus der Grafik kommen → dann ist der Text zwar „klar“, aber potenziell falsch/überinterpretiert (siehe unten „Meaningfulness/Correctness“).

2. Conciseness (Knappheit)

Tendenz: LLM-Texte sind in deinem Dokument deutlich weniger knapp (Conciseness oft 1–2). Gold-Standard ist meistens knapper, weil er weniger „Füllstruktur“ braucht und stärker priorisiert.

Beispiel, wo LLM wenig knapp ist:
Beim Tourismus-Japan-Chart wird sehr ausführlich erklärt (inkl. Ländergruppen-Vergleich + viele Zahlen), und Conciseness wird teils mit 1 bewertet.

Beispiel, wo LLM relativ knapp bleibt (für LLM-Verhältnisse):
Einige Texte bleiben bei den wichtigsten Stützpunkten (Start/Ende + grobe Zwischenwerte) und kommen auf Conciseness 2.

Interpretation: LLMs „kaufen“ sich Verständlichkeit oft durch Länge. Manuelle Gold-Standards machen eher eine Informations-Diät: wenig, aber gezielt.

3. Meaningfulness (Sinnhaftigkeit/Nutzen)

Hier wird’s spannend: Gold-Standard wirkt oft meaningful, weil er die Aussage der Grafik priorisiert (Pattern/Ranking/Takeaway), statt nur Achsen + Zahlen „abzuschreiben“. LLMs sind manchmal meaningful, aber oft auch zahlenlastig oder interpretieren zu viel.

Wo LLM meaningful sein kann
Wenn ein LLM neben Zahlen auch ein klares Muster nennt („kontinuierlicher Anstieg“, „Nigeria größter Zuwachs“), ist das für Nutzende hilfreich.

Wo LLM weniger meaningful wird (oder riskant)

1. Über-Interpretation / externe Begründungen:
   Beim MWSt-Chart tauchen Erklärungen wie „zur Finanzierung der AHV / Mitte-Initiative“ auf. Das kann meaningful sein, wenn es sicher aus Titel/Untertitel/Quelle folgt – aber in deinen Bewertungen sieht man dazu teils Correctness-Abzüge (z.B. Correctness 3 oder 2).
   Meaningfulness leidet, wenn Nutzende falsche Ursachen „lernen“.

2. Zahlen ohne Priorisierung:
   LLM-Long-Descriptions listen oft sehr viele Werte. Das fühlt sich informativ an, ist aber nicht immer der beste Nutzen für Screenreader-User: Wichtig ist meist Trend + Extremwerte + wichtigste Unterschiede. (Dein Gold-Standard beim Kakao-Chart macht genau das.)

4) PC = Perceived Completeness (wahrgenommene Vollständigkeit)

Auffällig im Dokument: PC ist bei LLM-Texten extrem oft 5, selbst wenn „Completeness“ nur 2–3 ist. Das spricht dafür, dass Format + Länge „Vollständigkeit“ signalisieren, auch wenn inhaltlich etwas fehlt oder unsicher ist.

Wo PC eher zutrifft (hoch, plausibel):
Wenn ein Text Achsentyp + Zeitraum + wichtigste Werte/Extremwerte abdeckt und strukturiert ist, wirkt er vollständig (z.B. klarer Zeitverlauf 1995–2030 mit Kernwerten).
Beim Versicherungsprämien-Chart sind die Beschreibungen sehr systematisch (beide Reihen, Start-/Endwerte, Vergleich „in jedem Jahr höher“) → PC 5 wirkt gerechtfertigt.

Wo PC „zu hoch“ wirken kann (hoch, aber kritisch):
Wenn die Long description „viel sagt“, aber inhaltlich danebenliegt oder Interpretationen einschmuggelt, kann es sich vollständig anfühlen, obwohl es es nicht ist (Correctness-Abzüge bei einigen LLM-Varianten).
Wenn „Completeness“ niedrig bewertet ist, PC aber 5 bleibt, deutet das genau auf diese Diskrepanz hin: wahrgenommene statt tatsächliche Vollständigkeit.

Fazit zur Research Question

Clarity: LLM ≈ Gold-Standard (oft sehr gut), LLM gewinnt durch standardisierte Struktur.
Conciseness: Gold-Standard > LLM (LLM häufig zu lang).
Meaningfulness: Gold-Standard oft > LLM, weil manuelle Texte stärker priorisieren und weniger halluzinierte/unsichere Begründungen einbauen; LLM kann meaningful sein, kippt aber bei Überinterpretation oder Zahlenflut.
PC: LLM wirkt häufig „vollständig“ (PC hoch), auch wenn die inhaltliche Vollständigkeit/Correctness das nicht immer stützt.

Eine (wichtige) Rückfrage an dich: In deiner Arbeit – willst du „Meaningfulness“ eher als Task-Utility für Screenreader-User definieren (Pattern/Takeaway zuerst) oder eher als maximale Datenabdeckung (viele Zahlen)?


## Inter-generation consistency

Wie konsistent sind mehrere Alt-Texte, die aus demselben Prompt generiert werden?

### Grundbeobachtung aus dem Dokument

Für jedes Diagramm werden bis zu 10 LLM-generierte Alt-Texte aus demselben Prompt gezeigt, sortiert nach SBERT-Ähnlichkeit zum Gold-Standard. Formal teilen sie also:

* denselben Prompt
* dasselbe visuelle Input-Diagramm
* dasselbe Ziel (Alt-Text)

Trotzdem zeigen sich erkennbare qualitative Unterschiede in Struktur, Detailtiefe und Korrektheit. 



### 1.1 Hohe Konsistenz bei einfachen Diagrammtypen

Zeitreihen mit wenigen Linien oder einfache Balkendiagramme zeigen eine relativ hohe Inter-Generation Consistency.

Beispiel: CO₂-Emissionen (Index 1990 = 100)

* Mehrere Alt-Texte beschreiben konsistent:

  * Startwert nahe 100
  * Anstieg bis ca. 2007
  * Rückgang nach 2008
* Clarity und Completeness sind durchgehend hoch (meist 4–5).
* Unterschiede betreffen primär Formulierung, nicht Inhalt. 

Interpretation:
Das Modell ist bei klaren, eindimensionalen Trends stabil. Die semantische Kernaussage bleibt über Generationen hinweg gleich → hohe Konsistenz.



### 1.2 Sinkende Konsistenz bei komplexen oder gestapelten Diagrammen

Bei Diagrammen mit vielen Kategorien, Segmenten oder überlagernden Zeitreihen sinkt die Konsistenz deutlich.

Beispiel: UMA (Unbegleitete minderjährige Ausländer) nach Nationalität

* Einige Alt-Texte fokussieren stark auf syrische Staatsangehörige,
* andere nennen mehrere Nationalitäten,
* wieder andere verlieren sich in Detailaufzählungen.
* Correctness-Werte fallen teils auf 2, PC ebenfalls auf 2–3. 

Interpretation:
Die Generationen sind inhaltlich nicht stabil:
Was als „zentrale Aussage“ gilt, variiert von Run zu Run. Das ist ein klares Zeichen geringer Inter-Generation Consistency bei komplexem Input.



### 1.3 Kritischer Sonderfall: scheinbar konsistent, aber falsch

Besonders problematisch sind Fälle, in denen mehrere Generationen ähnlich formuliert, aber inhaltlich problematisch sind.

Beispiel: Zürich – Luft- vs. Wassertemperatur

* Mehrere Alt-Texte ähneln sich stark im Aufbau,
* aber enthalten laut Judge faktische Ungenauigkeiten (Correctness = 2).
* Die Konsistenz ist also formal hoch, inhaltlich jedoch nicht zuverlässig. 

Kritischer Punkt:
Inter-Generation Consistency ≠ Qualität.
Ein Modell kann konsistent falsche oder überinterpretierte Aussagen reproduzieren.



## 2) Wie gut erfasst SBERT semantische Diversität?

### 2.1 Was SBERT im Dokument tatsächlich tut

SBERT wird hier genutzt, um:

* LLM-Alt-Texte nach Ähnlichkeit zum Gold-Standard zu ranken
* die „Top-10 ähnlichsten“ Varianten auszuwählen 

Wichtig:
Das Dokument zeigt keine low-similarity Outputs. Semantische Diversität wird also vorab herausgefiltert.



### 2.2 SBERT erfasst formale Nähe – nicht zwingend semantische Qualität

Beispiel: Touristen-Ausgaben in Japan

* Eine Variante erhält PC = 5, obwohl:

  * Clarity = 2
  * Completeness = 2
  * Text strukturell „zerbrochen“ wirkt
* Trotzdem gehört sie zu den SBERT-top-gerankten Texten. 

Schlussfolgerung:
SBERT misst offenbar:

* ähnliche Wortwahl
* ähnliche Satzbausteine

aber nicht zuverlässig:

* Verständlichkeit
* inhaltliche Fokussierung
* sinnvolle Priorisierung



### 2.3 Semantische Unterschiede, die SBERT nicht sauber trennt

Im Dokument finden sich Alt-Texte, die:

* denselben Trend beschreiben,
* aber unterschiedliche narrative Schwerpunkte setzen
  (z. B. Rangfolge vs. absolute Werte vs. zeitliche Dynamik)

SBERT ordnet sie trotzdem sehr nah ein.

Beispiel: Kakaoproduktion Westafrika

* Einige Varianten betonen die Dominanz von Côte d’Ivoire,
* andere listen detaillierte Produktionsmengen aller Länder.
* Semantisch unterschiedliche Nutzbarkeit, aber ähnliche SBERT-Nähe. 

Interpretation:
SBERT ist blind für funktionale Diversität („Was hilft dem Nutzer?“), solange die inhaltlichen Tokens ähnlich bleiben.



## 3) Gesamtbewertung (kritisch zusammengefasst)

### Inter-generation consistency

* Hoch bei:

  * einfachen Zeitreihen
  * wenigen Kategorien
* Niedrig bei:

  * komplexen, gestapelten Diagrammen
  * vielen Kategorien
  * interpretativen Aussagen
* Konsistenz kann auch konsistent falsch sein.

### SBERT & semantische Diversität

* Gut geeignet für:

  * grobe Inhaltsnähe
  * Duplikaterkennung
* Unzureichend für:

  * qualitative Unterschiede
  * Nutzerrelevanz
  * Klarheit vs. Redundanz
* Die im Dokument gezeigte Pipeline reduziert Diversität systematisch, statt sie zu analysieren.



## Kernaussage für deine Research Question

Multiple Alt-Texte aus demselben Prompt sind nur dann konsistent, wenn das visuelle Signal einfach und eindeutig ist. Mit steigender Diagrammkomplexität nimmt die inhaltliche Varianz deutlich zu. Embedding-basierte Methoden wie SBERT erfassen primär lexikalisch-strukturelle Nähe, nicht jedoch funktionale oder qualitative semantische Diversität, und können daher Unterschiede in Klarheit, Korrektheit und Aussagefokus nur unzureichend abbilden.

Wenn du möchtest, formuliere ich dir daraus eine direkt zitierfähige Passage (Academic English oder Deutsch) oder eine kritische Methodensektion für deine Arbeit.


In [ ]:
# # ----------------------------
# # CONFIG
# # ----------------------------

# DB_PATH = "../chart_database.db"
# GENERATION_RUN_ID = 1
# CHART_IMAGE_ROOT = "../data/NZZ_original"  # {chart_id}/{chart_id}.png

# OUTPUT_DIR = Path("../data/report_out_run")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# IMAGES_DIR = OUTPUT_DIR / "images"
# IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# REPORT_QMD = OUTPUT_DIR / f"generation_run_{GENERATION_RUN_ID}.qmd"

# # Bildgrösse für PDF konsistent
# RESIZE_IMAGE = True
# IMAGE_TARGET_WIDTH_PX = 900

# # ----------------------------
# # DB HELPERS
# # ----------------------------

# def dict_factory(cursor, row):
#     return {col[0]: row[idx] for idx, col in enumerate(cursor.description)}

# def connect_db(db_path: str) -> sqlite3.Connection:
#     conn = sqlite3.connect(db_path)
#     conn.row_factory = dict_factory
#     return conn

# def fetch_all(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...]):
#     return conn.execute(sql, params).fetchall()

# def fetch_one(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...]):
#     return conn.execute(sql, params).fetchone()

# # ----------------------------
# # IMAGE HELPERS
# # ----------------------------

# def chart_png_path(chart_id: str) -> Path:
#     return Path(CHART_IMAGE_ROOT) / str(chart_id) / f"{chart_id}.png"

# def save_chart_to_report_folder(chart_id: str) -> Path:
#     src = chart_png_path(chart_id)
#     if not src.exists():
#         raise FileNotFoundError(f"Chart image not found: {src}")

#     dst = IMAGES_DIR / f"{chart_id}.png"

#     img = Image.open(src).convert("RGB")
#     if RESIZE_IMAGE:
#         w, h = img.size
#         if w > IMAGE_TARGET_WIDTH_PX:
#             new_h = int(h * (IMAGE_TARGET_WIDTH_PX / w))
#             img = img.resize((IMAGE_TARGET_WIDTH_PX, new_h), resample=Image.Resampling.LANCZOS)

#     img.save(dst, format="PNG")
#     return dst

# # ----------------------------
# # SQL
# # ----------------------------

# SQL_ALTS_FOR_RUN = """
# SELECT
#   a.id AS alt_text_id,
#   a.chart_id,
#   a.variant_no,
#   a.short_description_metadata,
#   a.short_description_overview,
#   a.long_description
# FROM alt_text a
# WHERE a.generation_run_id = ?
# ORDER BY a.chart_id;
# """

# SQL_LLM_MEAN_FOR_ALT = """
# SELECT
#   AVG(score_clarity) AS clarity,
#   AVG(score_completeness) AS completeness,
#   AVG(score_conciseness) AS conciseness,
#   AVG(score_preceived_completeness) AS perceived_completeness,
#   AVG(score_neutrality) AS neutrality,
#   AVG(score_correctness) AS correctness,
#   COUNT(*) AS n
# FROM llm_evaluation
# WHERE alt_text_id = ?;
# """

# SQL_HUMAN_MEAN_FOR_ALT = """
# SELECT
#   AVG(score_clarity) AS clarity,
#   AVG(score_conciseness) AS conciseness,
#   AVG(score_preceived_completeness) AS perceived_completeness,
#   AVG(score_neutrality) AS neutrality,
#   COUNT(*) AS n
# FROM people_evaluation
# WHERE alt_text_id = ?;
# """

# SQL_SBERT = """
# SELECT similarity_score
# FROM alt_text_similarity_to_gold_standard
# WHERE alt_text_id = ?
# LIMIT 1;
# """

# # ----------------------------
# # QUARTO / LATEX HELPERS
# # ----------------------------

# def esc_latex(s: Any) -> str:
#     if s is None:
#         return ""
#     s = str(s)
#     repl = {
#         "\\": r"\textbackslash{}",
#         "&": r"\&",
#         "%": r"\%",
#         "$": r"\$",
#         "#": r"\#",
#         "_": r"\_",
#         "{": r"\{",
#         "}": r"\}",
#         "~": r"\textasciitilde{}",
#         "^": r"\textasciicircum{}",
#     }
#     for k, v in repl.items():
#         s = s.replace(k, v)
#     return s

# def fmt_score(x: Any) -> str:
#     if x is None:
#         return "—"
#     try:
#         return f"{float(x):.2f}".rstrip("0").rstrip(".")
#     except Exception:
#         return "—"

# def fmt_score_md(x: Any, digits: int = 2) -> str:
#     if x is None:
#         return "—"
#     try:
#         return f"{float(x):.{digits}f}".rstrip("0").rstrip(".")
#     except Exception:
#         return "—"

# def write_header(fp, run_id: int):
#     fp.write(
# f"""---
# title: "Generation Run ID {run_id}"
# format:
#   pdf:
#     toc: true
#     number-sections: false
#     pdf-engine: xelatex
#     include-in-header:
#       text: |
#         \\usepackage{{array}}
#         \\usepackage{{graphicx}}

#         \\usepackage[a4paper,top=1.6cm,bottom=2.4cm,left=1.6cm,right=1.6cm,footskip=1.2cm]{{geometry}}

#         % stabile Fusszeile (Seitenzahl)
#         \\usepackage{{fancyhdr}}
#         \\pagestyle{{fancy}}
#         \\fancyhf{{}}
#         \\fancyfoot[C]{{\\thepage}}
#         \\renewcommand{{\\headrulewidth}}{{0pt}}
#         \\renewcommand{{\\footrulewidth}}{{0pt}}

#         % Arial-ähnlich (Helvetica)
#         \\usepackage{{helvet}}
#         \\renewcommand{{\\familydefault}}{{\\sfdefault}}

#         \\setlength{{\\parindent}}{{0pt}}
#         \\renewcommand{{\\arraystretch}}{{1.15}}

#         % kompakte Aufzählungen
#         \\usepackage{{enumitem}}
#         \\setlist[itemize]{{leftmargin=*,topsep=0pt,itemsep=0pt,parsep=0pt,partopsep=0pt}}



# execute:
#   echo: false
#   warning: false
#   message: false
# ---

# """
#     )

# def write_intro(fp, run_id: int):
#     fp.write("# Overview\n\n")
#     fp.write(
#         f"Generation Run ID {run_id} contains all alt texts that were generated for the second interview. "
#         "The texts were generated with Gemini using temperature = 1 (default).\n\n"
#     )

# def write_layout_example_table(fp, example_chart_id: str):
#     fp.write("# Layout example\n\n")
#     fp.write(
#         "The report uses a two-column table for each chart. The left column contains the generated alt text "
#         "(metadata, overview, long description) and the SBERT similarity to golden standard. The right column contains all ratings "
#         "from *LLM as Judge* and (if available) *Human (mean)*.\n\n"
#     )
#     fp.write("Example (one chart_id):\n\n")

#     fp.write("```{=latex}\n")
#     fp.write(r"\begin{tabular}{|p{0.25\linewidth}|p{0.70\linewidth}|}" "\n")
#     fp.write(r"\hline" "\n")
#     fp.write(r"\textbf{Area} & \textbf{What you will find} \\ " "\n")
#     fp.write(r"\hline" "\n")
#     fp.write(rf"Header row & \texttt{{chart\_id = {example_chart_id}}} \\ " "\n")
#     fp.write(r"Left column & Generated text \\ " "\n")
#     fp.write(r"Right column & SBERT similarity + LLM as Judge scores + Human (mean) scores \\ " "\n")
#     fp.write(r"Image row & Chart image (left cell) \\ " "\n")
#     fp.write(r"\hline" "\n")
#     fp.write(r"\end{tabular}" "\n")
#     fp.write("```\n\n")

 
# def write_box_for_chart(
#     fp,
#     chart_id: str,
#     image_rel: str,
#     generated: Dict[str, Any],
#     llm: Dict[str, Any],
#     human: Dict[str, Any],
#     sbert: Optional[float],
# ):
#     meta = esc_latex(generated.get("short_description_metadata"))
#     overview = esc_latex(generated.get("short_description_overview"))
#     longd = esc_latex(generated.get("long_description"))

#     # Left column: ONLY generated text
#     sbert_line = ""
#     if sbert is not None:
#         sbert_line = f"\\textbf{{SBERT similarity:}} {fmt_score(sbert)}\\\\[6pt]\n"

#     left_block = (
#         r"\textbf{Short description (metadata):}\\[2pt]" "\n"
#         + meta + r"\\[8pt]" "\n"
#         + r"\textbf{Short description (overview):}\\[2pt]" "\n"
#         + overview + r"\\[8pt]" "\n"
#         + r"\textbf{Long description:}\\[2pt]" "\n"
#         + longd + "\n"
#     )

#     # Right column: ALL ratings (LLM + Human)
#     llm_block = (
#         sbert_line + "\n"
#         r"\textbf{LLM as Judge}\\[2pt]" "\n"
#         r"\begin{itemize}" "\n"
#         rf"\item Clarity: {fmt_score(llm.get('clarity'))}" "\n"
#         rf"\item Completeness: {fmt_score(llm.get('completeness'))}" "\n"
#         rf"\item Conciseness: {fmt_score(llm.get('conciseness'))}" "\n"
#         rf"\item PC: {fmt_score(llm.get('perceived_completeness'))}" "\n"
#         rf"\item Neutrality: {fmt_score(llm.get('neutrality'))}" "\n"
#         rf"\item Correctness: {fmt_score(llm.get('correctness'))}" "\n"
#         r"\end{itemize}" "\n"
#     )

#     if (human.get("n") or 0) > 0:
#         human_block = (
#             r"\vspace{2pt}" "\n"
#             r"\textbf{Human (mean):}\\[4pt]" "\n"
#             r"\begin{itemize}" "\n"
#             rf"\item Clarity: {fmt_score(human.get('clarity'))}" "\n"
#             rf"\item Conciseness: {fmt_score(human.get('conciseness'))}" "\n"
#             rf"\item PC: {fmt_score(human.get('perceived_completeness'))}" "\n"
#             rf"\item Neutrality: {fmt_score(human.get('neutrality'))}" "\n"
#             r"\end{itemize}" "\n"
#         )
#     else:
#         human_block = (
#             r"\vspace{2pt}" "\n"
#             r"\textbf{Human (mean):}\\[4pt]—" "\n"
#         )

#     right_block = llm_block + human_block

#     fp.write("\n")
#     fp.write("```{=latex}\n")
#     fp.write(r"\begin{tabular}{|p{0.68\linewidth}|p{0.28\linewidth}|}" "\n")
#     fp.write(r"\hline" "\n")
#     fp.write(rf"\textbf{{{esc_latex(chart_id)}}} & \\ " "\n")
#     fp.write(r"\hline" "\n")
#     fp.write(
#         rf"\vspace{{0.4\baselineskip}}\includegraphics[width=\linewidth]{{{esc_latex(image_rel)}}}\vspace{{0.4\baselineskip}} & \\ "
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     fp.write(
#         r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + left_block +
#         r"\end{minipage}"
#         + " & "
#         + r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + right_block +
#         r"\end{minipage}"
#         + r"\\ " "\n"
#     )

#     fp.write(r"\hline" "\n")
#     fp.write(r"\end{tabular}" "\n")
#     fp.write("```\n\n")
#     fp.write("\\vspace{10pt}\n\n")

# def main():
#     conn = connect_db(DB_PATH)

#     alts = fetch_all(conn, SQL_ALTS_FOR_RUN, (GENERATION_RUN_ID,))
#     if not alts:
#         raise RuntimeError(f"No alt_text rows found for generation_run_id={GENERATION_RUN_ID}")

#     # Precompute overview rows for the table
#     overview_rows: List[Dict[str, Any]] = []
#     for a in alts:
#         alt_text_id = int(a["alt_text_id"])
#         chart_id = str(a["chart_id"])

#         llm = fetch_one(conn, SQL_LLM_MEAN_FOR_ALT, (alt_text_id,)) or {}
#         human = fetch_one(conn, SQL_HUMAN_MEAN_FOR_ALT, (alt_text_id,)) or {}

#         sbert_row = fetch_one(conn, SQL_SBERT, (alt_text_id,))
#         sbert = None
#         if sbert_row and sbert_row.get("similarity_score") is not None:
#             try:
#                 sbert = float(sbert_row["similarity_score"])
#             except Exception:
#                 sbert = None

#         overview_rows.append({
#             "chart_id": chart_id,
#             "sbert": sbert,
#             "llm_clarity": llm.get("clarity"),
#             "llm_completeness": llm.get("completeness"),
#             "llm_conciseness": llm.get("conciseness"),
#             "llm_pc": llm.get("perceived_completeness"),
#             "llm_neutrality": llm.get("neutrality"),
#             "llm_correctness": llm.get("correctness"),
#             "human_clarity": human.get("clarity"),
#             "human_conciseness": human.get("conciseness"),
#             "human_pc": human.get("perceived_completeness"),
#             "human_neutrality": human.get("neutrality"),
#         })

#     with open(REPORT_QMD, "w", encoding="utf-8") as fp:
#         write_header(fp, GENERATION_RUN_ID)

#         # 1) English intro text
#         write_intro(fp, GENERATION_RUN_ID)

#         # 2) Layout explanation table using one example chart_id
#         write_layout_example_table(fp, example_chart_id=str(alts[0]["chart_id"]))

#         # 3) Full boxes per chart_id
#         fp.write("# Detailed results\n\n")
#         for a in alts:
#             chart_id = str(a["chart_id"])
#             alt_text_id = int(a["alt_text_id"])

#             img_out = save_chart_to_report_folder(chart_id)
#             image_rel = f"images/{img_out.name}"

#             llm = fetch_one(conn, SQL_LLM_MEAN_FOR_ALT, (alt_text_id,)) or {}
#             human = fetch_one(conn, SQL_HUMAN_MEAN_FOR_ALT, (alt_text_id,)) or {}

#             sbert_row = fetch_one(conn, SQL_SBERT, (alt_text_id,))
#             sbert = None
#             if sbert_row and sbert_row.get("similarity_score") is not None:
#                 try:
#                     sbert = float(sbert_row["similarity_score"])
#                 except Exception:
#                     sbert = None

#             write_box_for_chart(
#                 fp=fp,
#                 chart_id=chart_id,
#                 image_rel=image_rel,
#                 generated=a,
#                 llm=llm,
#                 human=human,
#                 sbert=sbert,
#             )

#     conn.close()
#     print(f"Wrote Quarto report: {REPORT_QMD}")

# if __name__ == "__main__":
#     main()


In [ ]:
# # ----------------------------
# # CONFIG
# # ----------------------------

# DB_PATH = "../chart_database.db"
# GENERATION_RUN_ID = 1
# CHART_IMAGE_ROOT = "../data/NZZ_original"  # {chart_id}/{chart_id}.png

# OUTPUT_DIR = Path("../data/report_out_run")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# IMAGES_DIR = OUTPUT_DIR / "images"
# IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# REPORT_QMD = OUTPUT_DIR / f"generation_run_{GENERATION_RUN_ID}.qmd"

# # Bildgrösse für PDF konsistent
# RESIZE_IMAGE = True
# IMAGE_TARGET_WIDTH_PX = 900

# # ----------------------------
# # DB HELPERS
# # ----------------------------

# def dict_factory(cursor, row):
#     return {col[0]: row[idx] for idx, col in enumerate(cursor.description)}

# def connect_db(db_path: str) -> sqlite3.Connection:
#     conn = sqlite3.connect(db_path)
#     conn.row_factory = dict_factory
#     return conn

# def fetch_all(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...]):
#     return conn.execute(sql, params).fetchall()

# def fetch_one(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...]):
#     return conn.execute(sql, params).fetchone()

# # ----------------------------
# # IMAGE HELPERS
# # ----------------------------

# def chart_png_path(chart_id: str) -> Path:
#     return Path(CHART_IMAGE_ROOT) / str(chart_id) / f"{chart_id}.png"

# def save_chart_to_report_folder(chart_id: str) -> Path:
#     src = chart_png_path(chart_id)
#     if not src.exists():
#         raise FileNotFoundError(f"Chart image not found: {src}")

#     dst = IMAGES_DIR / f"{chart_id}.png"

#     img = Image.open(src).convert("RGB")
#     if RESIZE_IMAGE:
#         w, h = img.size
#         if w > IMAGE_TARGET_WIDTH_PX:
#             new_h = int(h * (IMAGE_TARGET_WIDTH_PX / w))
#             img = img.resize((IMAGE_TARGET_WIDTH_PX, new_h), resample=Image.Resampling.LANCZOS)

#     img.save(dst, format="PNG")
#     return dst

# # ----------------------------
# # SQL
# # ----------------------------

# SQL_ALTS_FOR_RUN = """
# SELECT
#   a.id AS alt_text_id,
#   a.chart_id,
#   a.variant_no,
#   a.short_description_metadata,
#   a.short_description_overview,
#   a.long_description
# FROM alt_text a
# WHERE a.generation_run_id = ?;
# """

# SQL_LLM_MEAN_FOR_ALT = """
# SELECT
#   AVG(score_clarity) AS clarity,
#   AVG(score_completeness) AS completeness,
#   AVG(score_conciseness) AS conciseness,
#   AVG(score_preceived_completeness) AS perceived_completeness,
#   AVG(score_neutrality) AS neutrality,
#   AVG(score_correctness) AS correctness,
#   COUNT(*) AS n
# FROM llm_evaluation
# WHERE alt_text_id = ?;
# """

# SQL_LLM_ALL_FOR_ALT = """
# SELECT
#   no_eval,
#   score_clarity, reasoning_clarity,
#   score_completeness, reasoning_completeness,
#   score_conciseness, reasoning_conciseness,
#   score_preceived_completeness, reasoning_preceived_completeness,
#   score_neutrality, reasoning_neutrality,
#   score_correctness, reasoning_correctness
# FROM llm_evaluation
# WHERE alt_text_id = ?;
# """


# SQL_HUMAN_MEAN_FOR_ALT = """
# SELECT
#   AVG(score_clarity) AS clarity,
#   AVG(score_conciseness) AS conciseness,
#   AVG(score_preceived_completeness) AS perceived_completeness,
#   AVG(score_neutrality) AS neutrality,
#   COUNT(*) AS n
# FROM people_evaluation
# WHERE alt_text_id = ?;
# """

# SQL_SBERT = """
# SELECT similarity_score
# FROM alt_text_similarity_to_gold_standard
# WHERE alt_text_id = ?
# LIMIT 1;
# """

# # ----------------------------
# # QUARTO / LATEX HELPERS
# # ----------------------------

# def esc_latex(s: Any) -> str:
#     if s is None:
#         return ""
#     s = str(s)
#     repl = {
#         "\\": r"\textbackslash{}",
#         "&": r"\&",
#         "%": r"\%",
#         "$": r"\$",
#         "#": r"\#",
#         "_": r"\_",
#         "{": r"\{",
#         "}": r"\}",
#         "~": r"\textasciitilde{}",
#         "^": r"\textasciicircum{}",
#     }
#     for k, v in repl.items():
#         s = s.replace(k, v)
#     return s

# def fmt_score(x: Any) -> str:
#     if x is None:
#         return "—"
#     try:
#         return f"{float(x):.2f}".rstrip("0").rstrip(".")
#     except Exception:
#         return "—"

# def fmt_score_md(x: Any, digits: int = 2) -> str:
#     if x is None:
#         return "—"
#     try:
#         return f"{float(x):.{digits}f}".rstrip("0").rstrip(".")
#     except Exception:
#         return "—"
    

# def fmt_reasonings(reasonings: List[str], max_items: int = 3) -> str:
#     """Join up to max_items reasonings into one LaTeX-safe line."""
#     reasonings = [r.strip() for r in reasonings if r and str(r).strip()]
#     if not reasonings:
#         return "—"
#     shown = reasonings[:max_items]
#     tail = ""
#     if len(reasonings) > max_items:
#         tail = f" (… +{len(reasonings) - max_items} more)"
#     return esc_latex(" | ".join(shown) + tail)

# def smaller_reasoning(s: str, delta_pt: int = 1) -> str:
#     """
#     Reduce font size by delta_pt (default: 1pt) with safe line spacing.
#     """
#     return rf"{{\fontsize{{\dimexpr\f@size pt-{delta_pt}pt}}{{\baselineskip}}\selectfont {s}}}"



# def collect_llm_reasonings(llm_rows: List[Dict[str, Any]]) -> Dict[str, List[str]]:
#     out = {
#         "clarity": [],
#         "completeness": [],
#         "conciseness": [],
#         "perceived_completeness": [],
#         "neutrality": [],
#         "correctness": [],
#     }
#     for r in llm_rows or []:
#         out["clarity"].append(r.get("reasoning_clarity"))
#         out["completeness"].append(r.get("reasoning_completeness"))
#         out["conciseness"].append(r.get("reasoning_conciseness"))
#         out["perceived_completeness"].append(r.get("reasoning_preceived_completeness"))
#         out["neutrality"].append(r.get("reasoning_neutrality"))
#         out["correctness"].append(r.get("reasoning_correctness"))
#     return out


# def write_header(fp, run_id: int):
#     fp.write(
# f"""---
# title: "Generation Run ID {run_id}"
# format:
#   pdf:
#     toc: true
#     number-sections: false
#     pdf-engine: xelatex
#     include-in-header:
#       text: |
#         \\usepackage{{array}}
#         \\usepackage{{graphicx}}

#         \\usepackage[a4paper,top=1.6cm,bottom=2.4cm,left=1.6cm,right=1.6cm,footskip=1.2cm]{{geometry}}

#         % stabile Fusszeile (Seitenzahl)
#         \\usepackage{{fancyhdr}}
#         \\pagestyle{{fancy}}
#         \\fancyhf{{}}
#         \\fancyfoot[C]{{\\thepage}}
#         \\renewcommand{{\\headrulewidth}}{{0pt}}
#         \\renewcommand{{\\footrulewidth}}{{0pt}}

#         % Sans Serif with real bold
#         \\usepackage{{fontspec}}
#         \\setmainfont{{TeX Gyre Heros}}


#         \\renewcommand{{\\familydefault}}{{\\sfdefault}}

#         \\setlength{{\\parindent}}{{0pt}}
#         \\renewcommand{{\\arraystretch}}{{1.15}}

#         % kompakte Aufzählungen
#         \\usepackage{{enumitem}}
#         \\setlist[itemize]{{leftmargin=*,topsep=0pt,itemsep=0pt,parsep=0pt,partopsep=0pt}}



# execute:
#   echo: false
#   warning: false
#   message: false
# ---

# """
#     )

# def write_intro(fp, run_id: int):
#     fp.write("# Overview\n\n")
#     fp.write(
#         f"Generation Run ID {run_id} contains all alt texts that were generated for the second interview. "
#         "The texts were generated with Gemini using temperature = 1 (default).\n\n"
#     )

# def write_layout_example_table(fp, example_chart_id: str):
#     fp.write("# Layout\n\n")
#     fp.write(
#         "The report uses a table for each chart. The first row contains the chart id. The seccond row contains the chart."
#         "In the following row is the Alt-text (metadata, overview, long description) generated by Gemini with default temperature 1.0. In the last row are all ratings:\n\n" \
#         "- *SBERT similarity* to golden standard.\n\n" \
#         "- *LLM as Judge (1-5)*\n\n"
#         "- *Human (mean)* (if available).\n\n"
#     )
 
# # ----------------------------
# # WRITE BOX (FIXED LaTeX)
# # ----------------------------

# def write_box_for_chart(
#     fp,
#     chart_id: str,
#     image_rel: str,
#     generated: Dict[str, Any],
#     llm_mean: Dict[str, Any],
#     human_mean: Dict[str, Any],
#     sbert: Optional[float],
#     llm_rows: List[Dict[str, Any]],
# ):
#     meta = esc_latex(generated.get("short_description_metadata"))
#     overview = esc_latex(generated.get("short_description_overview"))
#     longd = esc_latex(generated.get("long_description"))

#     # Reasonings (aggregated)
#     llm_reasons = collect_llm_reasonings(llm_rows)
#     n_llm = len(llm_rows or [])

#     # IMPORTANT: never use \\ inside a tabular cell (it can terminate the table row)
#     alt_block = (
#         r"\textbf{Short description (metadata):} "
#         + meta + r"\par\vspace{6pt}"
#         + r"\textbf{Short description (overview):} "
#         + overview + r"\par\vspace{6pt}"
#         + r"\textbf{Long description:} "
#         + longd + r"\par\vspace{6pt}"
#     )



#     # SBERT
#     sbert_line = f"{fmt_score(sbert)}" if sbert is not None else "—"

#     # Human block
#     if (human_mean.get("n") or 0) > 0:
#         human_block = (
#             r"\textbf{Human (mean=5):}\par"
#             r"\begin{itemize}"
#             rf"\item Clarity: {fmt_score(human_mean.get('clarity'))}"
#             rf"\item Conciseness: {fmt_score(human_mean.get('conciseness'))}"
#             rf"\item PC: {fmt_score(human_mean.get('perceived_completeness'))}"
#             rf"\item Neutrality: {fmt_score(human_mean.get('neutrality'))}"
#             r"\end{itemize}"
#         )
#     else:
#         human_block = r"\textbf{Human (mean):} —"

#     # LLM block incl. reasoning summary
#     llm_block = (
#         rf"\textbf{{LLM as Judge (n=1):}}\par "
#         r"\begin{itemize}"
#         rf"\item Clarity: {fmt_score(llm_mean.get('clarity'))}\par \textit{{Reasoning:}} {fmt_reasonings(llm_reasons['clarity'])}"
#         rf"\item Completeness: {fmt_score(llm_mean.get('completeness'))}\par \textit{{Reasoning:}} {fmt_reasonings(llm_reasons['completeness'])}"
#         rf"\item Conciseness: {fmt_score(llm_mean.get('conciseness'))}\par \textit{{Reasoning:}} {fmt_reasonings(llm_reasons['conciseness'])}"
#         rf"\item PC: {fmt_score(llm_mean.get('perceived_completeness'))}\par \textit{{Reasoning:}} {fmt_reasonings(llm_reasons['perceived_completeness'])}"
#         rf"\item Neutrality: {fmt_score(llm_mean.get('neutrality'))}\par \textit{{Reasoning:}} {fmt_reasonings(llm_reasons['neutrality'])}"
#         rf"\item Correctness: {fmt_score(llm_mean.get('correctness'))}\par \textit{{Reasoning:}} {fmt_reasonings(llm_reasons['correctness'])}"
#         r"\end{itemize}\vspace{6pt}"
#     )

#     scores_block = (
#         r"\textbf{SBERT similarity:} " + sbert_line + r"\par\vspace{6pt}"
#         + human_block
#         + r"\par\vspace{6pt}"
#         + llm_block
#     )


#     fp.write("\n")
#     fp.write("```{=latex}\n")
#     fp.write(r"\begin{tabular}{|p{\linewidth}|}" "\n")
#     fp.write(r"\hline" "\n")

#     # 1) chart_id
#     fp.write(rf"\textbf{{chart\_id = {esc_latex(chart_id)}}} \\" "\n")
#     fp.write(r"\hline" "\n")

#     # 2) image (wrap to avoid stray alignment issues)
#     fp.write(
#         r"\begin{minipage}[c]{\linewidth}\centering"
#         + rf"\vspace{{0.4\baselineskip}}\includegraphics[width=0.5\linewidth]{{{esc_latex(image_rel)}}}\vspace{{0.4\baselineskip}}"
#         + r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     # 3) alt text
#     fp.write(
#         r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + alt_block +
#         r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     # 4) scores + reasoning
#     fp.write(
#         r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + scores_block +
#         r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     fp.write(r"\end{tabular}" "\n")
#     fp.write("```\n\n")
#     fp.write("\\vspace{10pt}\n\n")


# # ----------------------------
# # MAIN LOOP (FIXED variable usage)
# # ----------------------------

# def main():
#     conn = connect_db(DB_PATH)

#     alts = fetch_all(conn, SQL_ALTS_FOR_RUN, (GENERATION_RUN_ID,))
#     if not alts:
#         raise RuntimeError(f"No alt_text rows found for generation_run_id={GENERATION_RUN_ID}")

#     with open(REPORT_QMD, "w", encoding="utf-8") as fp:
#         write_header(fp, GENERATION_RUN_ID)
#         write_intro(fp, GENERATION_RUN_ID)
#         write_layout_example_table(fp, example_chart_id=str(alts[0]["chart_id"]))

#         fp.write("# Detailed results\n\n")
#         for a in alts:
#             chart_id = str(a["chart_id"])
#             alt_text_id = int(a["alt_text_id"])

#             img_out = save_chart_to_report_folder(chart_id)
#             image_rel = f"images/{img_out.name}"

#             # IMPORTANT: fetch per-chart values here, and pass them through
#             llm_mean = fetch_one(conn, SQL_LLM_MEAN_FOR_ALT, (alt_text_id,)) or {}
#             human_mean = fetch_one(conn, SQL_HUMAN_MEAN_FOR_ALT, (alt_text_id,)) or {}
#             llm_rows = fetch_all(conn, SQL_LLM_ALL_FOR_ALT, (alt_text_id,)) or []

#             sbert_row = fetch_one(conn, SQL_SBERT, (alt_text_id,))
#             sbert = None
#             if sbert_row and sbert_row.get("similarity_score") is not None:
#                 try:
#                     sbert = float(sbert_row["similarity_score"])
#                 except Exception:
#                     sbert = None

#             write_box_for_chart(
#                 fp=fp,
#                 chart_id=chart_id,
#                 image_rel=image_rel,
#                 generated=a,
#                 llm_mean=llm_mean,
#                 human_mean=human_mean,
#                 sbert=sbert,
#                 llm_rows=llm_rows,
#             )

#     conn.close()
#     print(f"Wrote Quarto report: {REPORT_QMD}")

# if __name__ == "__main__":
#     main()


In [ ]:
# import sqlite3
# from pathlib import Path
# from typing import Any, Dict, List, Optional, Tuple

# from PIL import Image

# # ----------------------------
# # CONFIG
# # ----------------------------

# DB_PATH = "../chart_database.db"

# # Optional: nur Gold-Standards eines bestimmten Judge-Modells
# # -> None = alle Gold-Standards
# JUDGE_MODEL_ID_FILTER: Optional[int] = None

# CHART_IMAGE_ROOT = "../data/NZZ_original"  # {chart_id}/{chart_id}.png

# OUTPUT_DIR = Path("../outputs/report_out_gold_standard")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# IMAGES_DIR = OUTPUT_DIR / "images"
# IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# REPORT_QMD = OUTPUT_DIR / "gold_standard_alt_text.qmd"

# # Bildgrösse für PDF konsistent
# RESIZE_IMAGE = True
# IMAGE_TARGET_WIDTH_PX = 900


# # ----------------------------
# # DB HELPERS
# # ----------------------------

# def dict_factory(cursor, row):
#     return {col[0]: row[idx] for idx, col in enumerate(cursor.description)}

# def connect_db(db_path: str) -> sqlite3.Connection:
#     conn = sqlite3.connect(db_path)
#     conn.row_factory = dict_factory
#     return conn

# def fetch_all(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...] = ()):
#     return conn.execute(sql, params).fetchall()

# def fetch_one(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...] = ()):
#     return conn.execute(sql, params).fetchone()


# # ----------------------------
# # IMAGE HELPERS
# # ----------------------------

# def chart_png_path(chart_id: str) -> Path:
#     return Path(CHART_IMAGE_ROOT) / str(chart_id) / f"{chart_id}.png"

# def save_chart_to_report_folder(chart_id: str) -> Path:
#     src = chart_png_path(chart_id)
#     if not src.exists():
#         raise FileNotFoundError(f"Chart image not found: {src}")

#     dst = IMAGES_DIR / f"{chart_id}.png"

#     img = Image.open(src).convert("RGB")
#     if RESIZE_IMAGE:
#         w, h = img.size
#         if w > IMAGE_TARGET_WIDTH_PX:
#             new_h = int(h * (IMAGE_TARGET_WIDTH_PX / w))
#             img = img.resize((IMAGE_TARGET_WIDTH_PX, new_h), resample=Image.Resampling.LANCZOS)

#     img.save(dst, format="PNG")
#     return dst


# # ----------------------------
# # SQL
# # ----------------------------

# SQL_GOLD_ALL = """
# SELECT
#   g.id AS gold_id,
#   g.chart_id,

#   g.short_description_metadata,
#   g.short_description_overview,
#   g.long_description,

#   g.tokens_short_description_metadata,
#   g.tokens_short_description_overview,
#   g.tokens_long_description,

#   g.embedding_model_id,
#   g.judge_model_id,

#   g.score_clarity,
#   g.reason_clarity,
#   g.score_completeness,
#   g.reason_completeness,
#   g.score_conciseness,
#   g.reason_conciseness,
#   g.score_preceived_completeness,
#   g.reason_preceived_completeness,
#   g.score_neutrality,
#   g.reason_neutrality,
#   g.score_correctness,
#   g.reason_correctness

# FROM gold_standard_alt_text g
# ORDER BY g.chart_id, g.id;
# """

# SQL_GOLD_FILTER_BY_JUDGE = """
# SELECT
#   g.id AS gold_id,
#   g.chart_id,

#   g.short_description_metadata,
#   g.short_description_overview,
#   g.long_description,

#   g.tokens_short_description_metadata,
#   g.tokens_short_description_overview,
#   g.tokens_long_description,

#   g.embedding_model_id,
#   g.judge_model_id,

#   g.score_clarity,
#   g.reason_clarity,
#   g.score_completeness,
#   g.reason_completeness,
#   g.score_conciseness,
#   g.reason_conciseness,
#   g.score_preceived_completeness,
#   g.reason_preceived_completeness,
#   g.score_neutrality,
#   g.reason_neutrality,
#   g.score_correctness,
#   g.reason_correctness

# FROM gold_standard_alt_text g
# WHERE g.judge_model_id = ?
# ORDER BY g.chart_id, g.id;
# """


# # ----------------------------
# # QUARTO / LATEX HELPERS
# # ----------------------------

# def esc_latex(s: Any) -> str:
#     if s is None:
#         return ""
#     s = str(s)
#     repl = {
#         "\\": r"\textbackslash{}",
#         "&": r"\&",
#         "%": r"\%",
#         "$": r"\$",
#         "#": r"\#",
#         "_": r"\_",
#         "{": r"\{",
#         "}": r"\}",
#         "~": r"\textasciitilde{}",
#         "^": r"\textasciicircum{}",
#     }
#     for k, v in repl.items():
#         s = s.replace(k, v)
#     return s

# def fmt_score(x: Any) -> str:
#     if x is None:
#         return "—"
#     try:
#         return f"{float(x):.2f}".rstrip("0").rstrip(".")
#     except Exception:
#         return "—"

# def fmt_int(x: Any) -> str:
#     if x is None:
#         return "—"
#     try:
#         return str(int(x))
#     except Exception:
#         return "—"

# def fmt_reason(x: Any) -> str:
#     x = ("" if x is None else str(x)).strip()
#     return esc_latex(x) if x else "—"


# def write_header(fp, judge_filter: Optional[int]):
#     subtitle = "all judge models" if judge_filter is None else f"judge_model_id = {judge_filter}"
#     fp.write(
# f"""---
# title: "Gold Standard Alt Text Report"
# subtitle: "{subtitle}"
# format:
#   pdf:
#     toc: true
#     number-sections: false
#     pdf-engine: xelatex
#     include-in-header:
#       text: |
#         \\usepackage{{array}}
#         \\usepackage{{graphicx}}

#         \\usepackage[a4paper,top=1.6cm,bottom=2.4cm,left=1.6cm,right=1.6cm,footskip=1.2cm]{{geometry}}

#         % stabile Fusszeile (Seitenzahl)
#         \\usepackage{{fancyhdr}}
#         \\pagestyle{{fancy}}
#         \\fancyhf{{}}
#         \\fancyfoot[C]{{\\thepage}}
#         \\renewcommand{{\\headrulewidth}}{{0pt}}
#         \\renewcommand{{\\footrulewidth}}{{0pt}}

#         % Sans Serif with real bold
#         \\usepackage{{fontspec}}
#         \\setmainfont{{TeX Gyre Heros}}
#         \\renewcommand{{\\familydefault}}{{\\sfdefault}}

#         \\setlength{{\\parindent}}{{0pt}}
#         \\renewcommand{{\\arraystretch}}{{1.15}}

#         % kompakte Aufzählungen
#         \\usepackage{{enumitem}}
#         \\setlist[itemize]{{leftmargin=*,topsep=0pt,itemsep=0pt,parsep=0pt,partopsep=0pt}}

# execute:
#   echo: false
#   warning: false
#   message: false
# ---

# """
#     )


# def write_intro(fp, judge_filter: Optional[int]):
#     fp.write("# Overview\n\n")
#     if judge_filter is None:
#         fp.write("This report contains all entries from `gold_standard_alt_text`.\n\n")
#     else:
#         fp.write(
#             f"This report contains entries from `gold_standard_alt_text` with "
#             f"`judge_model_id = {judge_filter}`.\n\n"
#         )

#     fp.write(
#         "For each chart, the report outputs a table containing:\n\n"
#         "- chart_id\n"
#         "- Chart image\n"
#         "- Gold alt text (metadata / overview / long description)\n"
#         "- Judge scores (1–5) including reasons (if available)\n"
#     )


# def write_box_for_chart_gold(
#     fp,
#     chart_id: str,
#     image_rel: str,
#     row: Dict[str, Any],
# ):
#     meta = esc_latex(row.get("short_description_metadata"))
#     overview = esc_latex(row.get("short_description_overview"))
#     longd = esc_latex(row.get("long_description"))

#     alt_block = (
#         r"\textbf{Short description (metadata):} " + meta + r"\par\vspace{6pt}"
#         r"\textbf{Short description (overview):} " + overview + r"\par\vspace{6pt}"
#         r"\textbf{Long description:} " + longd + r"\par\vspace{6pt}"
#     )

#     # Scores + Reasons (Gold Standard enthält sie schon)
#     scores_block = (
#         r"\textbf{Judge Scores (1–5):}\par"
#         r"\begin{itemize}"
#         rf"\item Clarity: {fmt_score(row.get('score_clarity'))}\par \textit{{Reason:}} {fmt_reason(row.get('reason_clarity'))}"
#         rf"\item Completeness: {fmt_score(row.get('score_completeness'))}\par \textit{{Reason:}} {fmt_reason(row.get('reason_completeness'))}"
#         rf"\item Conciseness: {fmt_score(row.get('score_conciseness'))}\par \textit{{Reason:}} {fmt_reason(row.get('reason_conciseness'))}"
#         rf"\item PC: {fmt_score(row.get('score_preceived_completeness'))}\par \textit{{Reason:}} {fmt_reason(row.get('reason_preceived_completeness'))}"
#         rf"\item Neutrality: {fmt_score(row.get('score_neutrality'))}\par \textit{{Reason:}} {fmt_reason(row.get('reason_neutrality'))}"
#         rf"\item Correctness: {fmt_score(row.get('score_correctness'))}\par \textit{{Reason:}} {fmt_reason(row.get('reason_correctness'))}"
#         r"\end{itemize}\vspace{6pt}"
#     )

#     fp.write("\n")
#     fp.write("```{=latex}\n")
#     fp.write(r"\begin{tabular}{|p{\linewidth}|}" "\n")
#     fp.write(r"\hline" "\n")

#     # 1) header line
#     fp.write(rf"\textbf{{{esc_latex(chart_id)}}} \\" "\n")
#     fp.write(r"\hline" "\n")

#     # 2) image
#     fp.write(
#         r"\begin{minipage}[c]{\linewidth}\centering"
#         + rf"\vspace{{0.4\baselineskip}}\includegraphics[width=0.5\linewidth]{{{esc_latex(image_rel)}}}\vspace{{0.4\baselineskip}}"
#         + r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     # 3) alt text
#     fp.write(
#         r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + alt_block +
#         r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     # 4) scores
#     fp.write(
#         r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + scores_block +
#         r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")


#     fp.write(r"\end{tabular}" "\n")
#     fp.write("```\n\n")
#     fp.write("\\vspace{10pt}\n\n")


# # ----------------------------
# # MAIN
# # ----------------------------

# def main():
#     conn = connect_db(DB_PATH)

#     if JUDGE_MODEL_ID_FILTER is None:
#         rows = fetch_all(conn, SQL_GOLD_ALL)
#     else:
#         rows = fetch_all(conn, SQL_GOLD_FILTER_BY_JUDGE, (JUDGE_MODEL_ID_FILTER,))

#     if not rows:
#         flt = "ALL" if JUDGE_MODEL_ID_FILTER is None else str(JUDGE_MODEL_ID_FILTER)
#         raise RuntimeError(f"No gold_standard_alt_text rows found (judge_model_id_filter={flt})")

#     with open(REPORT_QMD, "w", encoding="utf-8") as fp:
#         write_header(fp, JUDGE_MODEL_ID_FILTER)
#         write_intro(fp, JUDGE_MODEL_ID_FILTER)

#         fp.write("# Detailed results\n\n")
#         for r in rows:
#             chart_id = str(r["chart_id"])

#             img_out = save_chart_to_report_folder(chart_id)
#             image_rel = f"images/{img_out.name}"

#             write_box_for_chart_gold(
#                 fp=fp,
#                 chart_id=chart_id,
#                 image_rel=image_rel,
#                 row=r,
#             )

#     conn.close()
#     print(f"Wrote Quarto report: {REPORT_QMD}")

# if __name__ == "__main__":
#     main()


In [ ]:
# import sqlite3
# from pathlib import Path
# from typing import Any, Dict, List, Optional, Tuple
# from PIL import Image
# from collections import defaultdict

# # ----------------------------
# # CONFIG
# # ----------------------------

# DB_PATH = "../chart_database.db"
# GENERATION_RUN_ID = 4
# CHART_IMAGE_ROOT = "../data/NZZ_original"  # {chart_id}/{chart_id}.png

# OUTPUT_DIR = Path("../data/report_out_run")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# IMAGES_DIR = OUTPUT_DIR / "images"
# IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# REPORT_QMD = OUTPUT_DIR / f"generation_run_{GENERATION_RUN_ID}.qmd"

# # Bildgrösse für PDF konsistent
# RESIZE_IMAGE = True
# IMAGE_TARGET_WIDTH_PX = 900

# # ----------------------------
# # DB HELPERS
# # ----------------------------

# def dict_factory(cursor, row):
#     return {col[0]: row[idx] for idx, col in enumerate(cursor.description)}

# def connect_db(db_path: str) -> sqlite3.Connection:
#     conn = sqlite3.connect(db_path)
#     conn.row_factory = dict_factory
#     return conn

# def fetch_all(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...]):
#     return conn.execute(sql, params).fetchall()

# def fetch_one(conn: sqlite3.Connection, sql: str, params: Tuple[Any, ...]):
#     return conn.execute(sql, params).fetchone()

# # ----------------------------
# # IMAGE HELPERS
# # ----------------------------

# def chart_png_path(chart_id: str) -> Path:
#     return Path(CHART_IMAGE_ROOT) / str(chart_id) / f"{chart_id}.png"

# def save_chart_to_report_folder(chart_id: str) -> Path:
#     src = chart_png_path(chart_id)
#     if not src.exists():
#         raise FileNotFoundError(f"Chart image not found: {src}")

#     dst = IMAGES_DIR / f"{chart_id}.png"

#     img = Image.open(src).convert("RGB")
#     if RESIZE_IMAGE:
#         w, h = img.size
#         if w > IMAGE_TARGET_WIDTH_PX:
#             new_h = int(h * (IMAGE_TARGET_WIDTH_PX / w))
#             img = img.resize((IMAGE_TARGET_WIDTH_PX, new_h), resample=Image.Resampling.LANCZOS)

#     img.save(dst, format="PNG")
#     return dst

# # ----------------------------
# # SQL
# # ----------------------------

# SQL_GOLD_STANDARD_FOR_CHART = """
# SELECT
#   short_description_metadata,
#   short_description_overview,
#   long_description
# FROM gold_standard_alt_text
# WHERE chart_id = ?
# LIMIT 1;
# """


# SQL_ALTS_FOR_RUN = """
# SELECT
#   a.id AS alt_text_id,
#   a.chart_id,
#   a.variant_no,
#   a.short_description_metadata,
#   a.short_description_overview,
#   a.long_description
# FROM alt_text a
# WHERE a.generation_run_id = ?;
# """

# SQL_LLM_MEAN_FOR_ALT = """
# SELECT
#   AVG(score_clarity) AS clarity,
#   AVG(score_completeness) AS completeness,
#   AVG(score_conciseness) AS conciseness,
#   AVG(score_preceived_completeness) AS perceived_completeness,
#   AVG(score_neutrality) AS neutrality,
#   AVG(score_correctness) AS correctness,
#   COUNT(*) AS n
# FROM llm_evaluation
# WHERE alt_text_id = ?;
# """

# SQL_SBERT = """
# SELECT similarity_score
# FROM alt_text_similarity_to_gold_standard
# WHERE alt_text_id = ?
# LIMIT 1;
# """

# # ----------------------------
# # QUARTO / LATEX HELPERS
# # ----------------------------



# def esc_latex(s: Any) -> str:
#     if s is None:
#         return ""
#     s = str(s)
#     repl = {
#         "\\": r"\textbackslash{}",
#         "&": r"\&",
#         "%": r"\%",
#         "$": r"\$",
#         "#": r"\#",
#         "_": r"\_",
#         "{": r"\{",
#         "}": r"\}",
#         "~": r"\textasciitilde{}",
#         "^": r"\textasciicircum{}",
#     }
#     for k, v in repl.items():
#         s = s.replace(k, v)
#     return s

# def fmt_score(x: Any) -> str:
#     if x is None:
#         return "—"
#     try:
#         return f"{float(x):.2f}".rstrip("0").rstrip(".")
#     except Exception:
#         return "—"

# def write_header(fp, run_id: int):
#     fp.write(
# f"""---
# title: "Generation Run ID {run_id}"
# format:
#   pdf:
#     toc: true
#     number-sections: false
#     pdf-engine: xelatex
#     include-in-header:
#       text: |
#         \\usepackage{{array}}
#         \\usepackage{{graphicx}}
#         \\usepackage[a4paper,top=1.6cm,bottom=2.4cm,left=1.6cm,right=1.6cm,footskip=1.2cm]{{geometry}}

#         % stabile Fusszeile (Seitenzahl)
#         \\usepackage{{fancyhdr}}
#         \\pagestyle{{fancy}}
#         \\fancyhf{{}}
#         \\fancyfoot[C]{{\\thepage}}
#         \\renewcommand{{\\headrulewidth}}{{0pt}}
#         \\renewcommand{{\\footrulewidth}}{{0pt}}

#         % Sans Serif with real bold
#         \\usepackage{{fontspec}}
#         \\setmainfont{{TeX Gyre Heros}}
#         \\renewcommand{{\\familydefault}}{{\\sfdefault}}

#         \\setlength{{\\parindent}}{{0pt}}
#         \\renewcommand{{\\arraystretch}}{{1.15}}

#         % kompakte Aufzählungen
#         \\usepackage{{enumitem}}
#         \\setlist[itemize]{{leftmargin=*,topsep=0pt,itemsep=0pt,parsep=0pt,partopsep=0pt}}

#         \\usepackage{{longtable}}


# execute:
#   echo: false
#   warning: false
#   message: false
# ---

# """
#     )

# def write_intro(fp, run_id: int):
#     fp.write("# Overview\n\n")
#     fp.write(
#         f"Generation Run ID {run_id}: Report pro Chart mit 10 Alt-Text-Varianten. "
#         "Die Varianten sind pro Chart nach SBERT (desc) sortiert; Charts sind nach bestem SBERT gerankt.\n\n"
#     )

# def write_layout_example_table(fp):
#     fp.write("# Layout\n\n")
#     fp.write(
#         "Pro Chart:\n\n"
#         "- Zeile 1: chart_id\n"
#         "- Zeile 2: Chart\n"
#         "- Zeile 3–12: Top-10 Alt-Texte (links), sortiert nach SBERT Similarity score (generated vs golden standard alt text) + Scores (rechts). Scores rechts enthalten SBERT sowie LLM-as-judge für 6 Kriterien.\n\n"
#     )


# # ----------------------------
# # WRITE BOX PER CHART (12 rows)
# # ----------------------------

# def build_gold_block(gs_row: Optional[Dict[str, Any]]) -> str:
#     if not gs_row:
#         return r"\textbf{Golden standard:} —"

#     meta = esc_latex(gs_row.get("short_description_metadata"))
#     overview = esc_latex(gs_row.get("short_description_overview"))
#     longd = esc_latex(gs_row.get("long_description"))

#     return (
#         r"\textbf{Golden standard}\par\vspace{4pt}"
#         r"\textbf{Short description (metadata):} " + meta + r"\par\vspace{4pt}"
#         r"\textbf{Short description (overview):} " + overview + r"\par\vspace{4pt}"
#         r"\textbf{Long description:} " + longd
#     )


# def build_alt_block(alt_row: Dict[str, Any]) -> str:
#     meta = esc_latex(alt_row.get("short_description_metadata"))
#     overview = esc_latex(alt_row.get("short_description_overview"))
#     longd = esc_latex(alt_row.get("long_description"))

#     # Kein \\ innerhalb von Zellen — nur \par verwenden
#     return (
#         r"\textbf{Short description (metadata):} " + meta + r"\par\vspace{4pt}"
#         r"\textbf{Short description (overview):} " + overview + r"\par\vspace{4pt}"
#         r"\textbf{Long description:} " + longd
#     )

# def build_scores_block(sbert: Optional[float], llm_mean: Dict[str, Any]) -> str:
#     n = llm_mean.get("n")
#     n_line = f"{int(n)}" if n is not None else "—"

#     return (
#         r"\textbf{SBERT:} " + fmt_score(sbert) + r"\par\vspace{4pt}"
#         r"\textbf{LLM as Judge:} " + r"\textit{(n=" + esc_latex(n_line) + r")}\par"
#         r"\begin{itemize}"
#         rf"\item Clarity: {fmt_score(llm_mean.get('clarity'))}"
#         rf"\item Completeness: {fmt_score(llm_mean.get('completeness'))}"
#         rf"\item Conciseness: {fmt_score(llm_mean.get('conciseness'))}"
#         rf"\item PC: {fmt_score(llm_mean.get('perceived_completeness'))}"
#         rf"\item Neutrality: {fmt_score(llm_mean.get('neutrality'))}"
#         rf"\item Correctness: {fmt_score(llm_mean.get('correctness'))}"
#         r"\end{itemize}"
#     )


# def write_box_for_chart_10(
#     fp,
#     placement: int,
#     chart_id: str,
#     image_rel: str,
#     gold_row: Optional[Dict[str, Any]],
#     items_sorted: List[Dict[str, Any]],
# ):

#     fp.write("\n")
#     fp.write("```{=latex}\n")

#     # 1) Row: placement + chart_id (1 column)
#     fp.write(r"\begin{longtable}{|p{\linewidth}|}" "\n")
#     fp.write(r"\hline" "\n")
#     fp.write(rf"\textbf{{{esc_latex(chart_id)}}} \\" "\n")
#     fp.write(r"\hline" "\n")

#     # 2) Row: image (1 column, half width)
#     fp.write(
#         r"\begin{minipage}[c]{\linewidth}\centering"
#         + rf"\vspace{{0.4\baselineskip}}\includegraphics[width=0.5\linewidth]{{{esc_latex(image_rel)}}}\vspace{{0.4\baselineskip}}"
#         + r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")
    

#     # 3) Row: golden standard (1 column)
#     gs_block = build_gold_block(gold_row)
#     fp.write(
#         r"\begin{minipage}[t]{\linewidth}\raggedright" "\n"
#         + gs_block +
#         r"\end{minipage} \\"
#         "\n"
#     )
#     fp.write(r"\hline" "\n")

#     fp.write(r"\end{longtable}" "\n")
#     fp.write("```\n\n")

#     # 4-13) Rows: 10 alt texts (2 columns 0.60 / 0.35)
#     fp.write("```{=latex}\n")
#     fp.write(r"\begin{longtable}{|p{0.75\linewidth}|p{0.22\linewidth}|}" "\n")
#     fp.write(r"\hline" "\n")

#     for i, it in enumerate(items_sorted, start=1):
#         alt_block = build_alt_block(it)
#         scores_block = build_scores_block(it.get("sbert"), it.get("llm_mean") or {})

#         left = (
#             rf"\textbf{{{i}. Alt text}}\par\vspace{{4pt}}"
#             + alt_block
#         )
#         right = scores_block

#         fp.write(r"\begin{minipage}[t]{\linewidth}\raggedright" "\n" + left + r"\end{minipage} &" "\n")
#         fp.write(r"\begin{minipage}[t]{\linewidth}\raggedright" "\n" + right + r"\end{minipage} \\" "\n")
#         fp.write(r"\hline" "\n")

#     fp.write(r"\end{longtable}" "\n")
#     fp.write("```\n\n")
#     fp.write("\\vspace{12pt}\n\n")

# # ----------------------------
# # MAIN LOOP
# # ----------------------------

# def main():
#     conn = connect_db(DB_PATH)

#     alts = fetch_all(conn, SQL_ALTS_FOR_RUN, (GENERATION_RUN_ID,))
#     if not alts:
#         raise RuntimeError(f"No alt_text rows found for generation_run_id={GENERATION_RUN_ID}")

#     # Enrich rows with SBERT + LLM mean, then group by chart_id
#     grouped: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

#     for a in alts:
#         chart_id = str(a["chart_id"])
#         alt_text_id = int(a["alt_text_id"])

#         # LLM mean
#         llm_mean = fetch_one(conn, SQL_LLM_MEAN_FOR_ALT, (alt_text_id,)) or {}

#         # SBERT
#         sbert_row = fetch_one(conn, SQL_SBERT, (alt_text_id,))
#         sbert = None
#         if sbert_row and sbert_row.get("similarity_score") is not None:
#             try:
#                 sbert = float(sbert_row["similarity_score"])
#             except Exception:
#                 sbert = None

#         a2 = dict(a)
#         a2["llm_mean"] = llm_mean
#         a2["sbert"] = sbert
#         grouped[chart_id].append(a2)

#     # Sort within each chart by SBERT desc (None last)
#     for cid in list(grouped.keys()):
#         grouped[cid].sort(key=lambda r: (r["sbert"] is None, -(r["sbert"] or -1e9)))

#         # Falls es mehr/ weniger als 10 gibt: robust machen
#         # gewünschtes Layout ist 10; wir nehmen Top-10 und ignorieren Rest
#         grouped[cid] = grouped[cid][:10]

#         if len(grouped[cid]) < 10:
#             # Optional: warnen/auffüllen — hier nur RuntimeError für striktes Layout
#             raise RuntimeError(f"chart_id={cid} has only {len(grouped[cid])} alt texts; expected 10.")

#     # Rank charts by best SBERT (desc), None last
#     chart_order = sorted(
#         grouped.keys(),
#         key=lambda cid: (
#             grouped[cid][0]["sbert"] is None,
#             -(grouped[cid][0]["sbert"] or -1e9)
#         )
#     )

#     with open(REPORT_QMD, "w", encoding="utf-8") as fp:
#         write_header(fp, GENERATION_RUN_ID)
#         write_intro(fp, GENERATION_RUN_ID)
#         write_layout_example_table(fp)

#         fp.write("# Detailed results\n\n")

#         for placement, chart_id in enumerate(chart_order, start=1):
#             img_out = save_chart_to_report_folder(chart_id)
#             image_rel = f"images/{img_out.name}"

#             gold_row = fetch_one(conn, SQL_GOLD_STANDARD_FOR_CHART, (chart_id,))

#             write_box_for_chart_10(
#                 fp=fp,
#                 placement=placement,
#                 chart_id=chart_id,
#                 image_rel=image_rel,
#                 gold_row=gold_row,
#                 items_sorted=grouped[chart_id],
#             )

#     conn.close()
#     print(f"Wrote Quarto report: {REPORT_QMD}")

# if __name__ == "__main__":
#     main()
